# Simulation Monte-Carlo, estimation des paramètres et tests d'adéquation : loi Burr-PTLEN

Ce notebook est la traduction en Python du script R, complétée par des **tests d'adéquation**
(Kolmogorov-Smirnov, Anderson-Darling, Cramér-von Mises) et leurs **p-values**.

| R | Python |
|---|---|
| `stats4::mle` (L-BFGS-B) | `scipy.optimize.minimize(method="L-BFGS-B")` |
| `runif`, `set.seed` | `numpy.random.default_rng(123).uniform` |
| `tibble`, `dplyr`, `ggplot2` | `numpy`, `matplotlib` |

Paramètres de la loi : $x > 0$ et $m, n, p, \lambda, \beta, c, k > 0$.

> Installation : `pip install numpy scipy matplotlib`

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, optimize
from scipy.integrate import quad

# Générateur aléatoire avec graine fixée (équivalent de set.seed(123))
# Remarque : R et NumPy ne tirent pas les mêmes nombres pour une même graine.
rng = np.random.default_rng(123)

# Noms des 7 paramètres, dans l'ordre utilisé partout dans le notebook
NOMS = ["m", "n", "p", "lambda", "beta", "c", "k"]

## 1. Méthode numérique : la bissection

On cherche une racine de $f$ sur $[a, b]$ en coupant l'intervalle en deux à chaque étape,
à condition que $f(a)$ et $f(b)$ soient de signes opposés.

In [ ]:
def bissection(f, a, b, tol=1e-6, max_iter=1000):
    fa = f(a)
    fb = f(b)

    # Il faut un changement de signe entre a et b
    if fa * fb > 0:
        raise ValueError("L'intervalle donné ne contient pas de changement de signe.")

    for i in range(max_iter):
        x_milieu = (a + b) / 2
        f_milieu = f(x_milieu)

        # Arrêt : f proche de 0 ou intervalle assez petit
        if abs(f_milieu) < tol or (b - a) < tol:
            return x_milieu

        # On garde la moitié qui contient le changement de signe
        if fa * f_milieu < 0:
            b, fb = x_milieu, f_milieu
        else:
            a, fa = x_milieu, f_milieu

    warnings.warn("La méthode de la bissection n'a pas convergé.")
    return np.nan

## 2. Fonction de répartition et densité

On note $A = (1 + x^c)^{-k}$ (survie de Burr XII), $G = \dfrac{1 - A}{A}$ et

$$
K(x) = 1 - \exp\!\big(-\lambda\, G^{\beta}\big), \qquad u = n\,\big(1 - K^{-p}\big) \le 0 .
$$

La fonction de répartition et la densité sont :

$$
F(x) = e^{m u}\,\big(2 - e^{u}\big)^{m}
$$

$$
f(x) = 2mnp\lambda\beta ck\; x^{c-1}(1+x^c)^{-k-1}\;
\frac{(1-A)^{\beta-1}}{A^{\beta+1}}\;
\frac{1-K}{K^{p+1}}\, e^{m u}\;
\big(1 - e^{u}\big)\big(2 - e^{u}\big)^{m-1}
$$

En Python, les fonctions sont **vectorisées** : elles acceptent un nombre ou un tableau `x`.
`np.errstate` coupe les avertissements de dépassement aux bords du support ($x \to 0$ ou $x \to \infty$).

In [ ]:
def burrPTLENcdf(x, m, n, p, lam, beta, c, k):
    x = np.asarray(x, dtype=float)
    with np.errstate(over="ignore", divide="ignore", invalid="ignore"):
        K = 1 - np.exp(-lam * ((1 - (1 + x**c) ** (-k)) / (1 + x**c) ** (-k)) ** beta)
        cdf = np.exp(m * n * (1 - 1 / K**p)) * (2 - np.exp(n * (1 - 1 / K**p))) ** m
    return cdf


def burrPTLENpdf(x, m, n, p, lam, beta, c, k):
    x = np.asarray(x, dtype=float)
    with np.errstate(over="ignore", divide="ignore", invalid="ignore"):
        K = 1 - np.exp(-lam * ((1 - (1 + x**c) ** (-k)) / (1 + x**c) ** (-k)) ** beta)
        term_bur1 = 2 * m * n * p * lam * beta * c * k * x ** (c - 1) * (1 + x**c) ** (-k - 1)
        term_bur2 = (1 - (1 + x**c) ** (-k)) ** (beta - 1) / ((1 + x**c) ** (-k)) ** (beta + 1)
        term_tl1 = (1 - K) / K ** (p + 1) * np.exp(m * n * (1 - 1 / K**p))
        term_tl2 = (1 - np.exp(n * (1 - 1 / K**p))) * (2 - np.exp(n * (1 - 1 / K**p))) ** (m - 1)
        return term_bur1 * term_bur2 * term_tl1 * term_tl2

### Ajout : log-densité numériquement stable

Dans les queues de la loi, la densité devient si petite qu'elle est arrondie à **0** en machine
(alors que $\log f$ reste un nombre fini) : la log-vraisemblance vaut alors $-\infty$ et l'optimisation échoue.
On calcule donc directement $\log f$, terme à terme :

$$
\log f = \log(2mnp\lambda\beta ck) + (c-1)\log x - (k+1)L + (\beta-1)\log(1-A) + (\beta+1)kL
- \lambda G^\beta - (p+1)\log K + m u + \log(1-e^u) + (m-1)\log(2-e^u)
$$

avec $L = \log(1 + x^c)$. Les fonctions `log1p` et `expm1` évitent les pertes de précision
(par exemple $1 - e^{-t}$ quand $t$ est minuscule). C'est cette version qu'utilise l'estimation.

In [ ]:
def burrPTLENlogpdf(x, m, n, p, lam, beta, c, k):
    x = np.asarray(x, dtype=float)
    with np.errstate(over="ignore", divide="ignore", invalid="ignore"):
        L = np.log1p(x**c)                          # log(1 + x^c)
        G = np.expm1(k * L)                         # (1 + x^c)^k - 1 = (1 - A) / A
        lG = lam * G**beta                          # lambda * G^beta
        # log K = log(1 - exp(-lG)), calculé précisément pour lG petit ou grand
        logK = np.where(lG > np.log(2), np.log1p(-np.exp(-lG)), np.log(-np.expm1(-lG)))
        u = -n * np.expm1(-p * logK)                # n (1 - K^(-p)) <= 0
        # log(1 - e^u) ; quand lG est très grand, -u ≈ n p exp(-lG)
        log_1_moins_eu = np.where(lG > 30, np.log(n * p) - lG, np.log(-np.expm1(u)))
        return (np.log(2 * m * n * p * lam * beta * c * k) + (c - 1) * np.log(x)
                - (k + 1) * L + (beta - 1) * np.log(-np.expm1(-k * L)) + (beta + 1) * k * L
                - lG - (p + 1) * logK + m * u + log_1_moins_eu
                + (m - 1) * np.log1p(-np.expm1(u)))

**Vérifications (ajout) :** la densité doit s'intégrer à 1 et être la dérivée de la fonction de répartition.

In [ ]:
# Paramètres "vrais" utilisés pour la simulation (ceux du script R)
theta_vrai = dict(m=2, n=1.4, p=3.1, lam=0.5, beta=1.5, c=3, k=1)

integrale, _ = quad(lambda x: burrPTLENpdf(x, **theta_vrai), 0, np.inf, limit=200)
print("Intégrale de la densité :", integrale)

x0, h = 1.3, 1e-6
derivee = (burrPTLENcdf(x0 + h, **theta_vrai) - burrPTLENcdf(x0 - h, **theta_vrai)) / (2 * h)
print("f(1.3) =", burrPTLENpdf(x0, **theta_vrai), " ; dF/dx(1.3) ≈", derivee)
print("exp(logpdf(1.3)) =", np.exp(burrPTLENlogpdf(x0, **theta_vrai)))

# Dans la queue gauche, la densité "directe" est arrondie à 0, pas la log-densité
for x in [0.5, 4]:
    print(f"x = {x}: pdf = {burrPTLENpdf(x, **theta_vrai)}, logpdf = {burrPTLENlogpdf(x, **theta_vrai):.2f}")

## 3. Simulation par inversion de la fonction de répartition

Méthode de la transformée inverse : si $U \sim \mathcal{U}(0,1)$, alors $X = F^{-1}(U)$ suit la loi de $F$.
Comme dans le script R, on résout $F(x) = U$ par bissection sur $[0{,}1 ; 100]$.

In [ ]:
def invert_cdf(U, m, n, p, lam, beta, c, k):
    f = lambda x: burrPTLENcdf(x, m, n, p, lam, beta, c, k) - U
    # L'intervalle [0.1, 100] est choisi en fonction du support attendu
    return bissection(f, a=0.1, b=100)


def simulate_loi(nb, m, n, p, lam, beta, c, k):
    U = rng.uniform(size=nb)
    # Équivalent de sapply : on inverse chaque U
    return np.array([invert_cdf(u, m, n, p, lam, beta, c, k) for u in U])

### Ajout : quantile explicite

$F$ s'inverse en fait **exactement**. En posant $v = e^{u} \in (0, 1]$, on a $F = \big(v(2 - v)\big)^m$, d'où :

$$
v = 1 - \sqrt{1 - U^{1/m}}, \quad
K = \Big(1 - \frac{\ln v}{n}\Big)^{-1/p}, \quad
G = \Big(\frac{-\ln(1 - K)}{\lambda}\Big)^{1/\beta}, \quad
x = \Big((1 + G)^{1/k} - 1\Big)^{1/c}.
$$

Cette formule est exacte, instantanée et ne dépend d'aucun intervalle de recherche.
On s'en sert pour **valider la bissection**, puis pour le bootstrap de la section 6, qui simule beaucoup d'échantillons.
Elle est écrite avec `log1p` / `expm1` : sinon, pour de petits $m$ ou de petits $U$, des différences comme
$1 - \sqrt{1 - \varepsilon}$ seraient arrondies à 0 et donneraient $x = 0$.

In [ ]:
def burrPTLENquantile(U, m, n, p, lam, beta, c, k):
    U = np.asarray(U, dtype=float)
    with np.errstate(over="ignore", divide="ignore", invalid="ignore"):
        v = -np.expm1(0.5 * np.log1p(-np.exp(np.log(U) / m)))   # 1 - sqrt(1 - U^(1/m))
        K = np.exp(-np.log1p(-np.log(v) / n) / p)                # (1 - ln(v)/n)^(-1/p)
        G = (-np.log1p(-K) / lam) ** (1 / beta)                  # (-ln(1-K)/lambda)^(1/beta)
        return np.expm1(np.log1p(G) / k) ** (1 / c)              # ((1+G)^(1/k) - 1)^(1/c)


# Comparaison bissection / formule exacte sur quelques valeurs de U
U_test = np.array([0.01, 0.25, 0.5, 0.75, 0.99])
x_bis = np.array([invert_cdf(u, **theta_vrai) for u in U_test])
x_exact = burrPTLENquantile(U_test, **theta_vrai)
print("bissection :", np.round(x_bis, 5))
print("exact      :", np.round(x_exact, 5))
print("écart max  :", np.max(np.abs(x_bis - x_exact)))

### Simulation de 250 observations

Histogramme des simulations, densité estimée (noyau, équivalent de `geom_density`) et densité théorique en rouge :
si les courbes se superposent, l'inversion est juste.

In [ ]:
simulation1 = simulate_loi(nb=250, **theta_vrai)

grille = np.linspace(simulation1.min(), simulation1.max(), 400)
kde = stats.gaussian_kde(simulation1)   # densité estimée par noyau

plt.figure(figsize=(8, 5))
plt.hist(simulation1, bins=np.arange(0, simulation1.max() + 0.2, 0.2), density=True,
         color="skyblue", edgecolor="black", label="histogramme")
plt.plot(grille, kde(grille), color="black", lw=2, label="densité estimée (noyau)")
plt.plot(grille, burrPTLENpdf(grille, **theta_vrai), color="red", lw=2, label="densité théorique")
plt.title("Histogramme des simulations")
plt.ylabel("densité")
plt.legend()
plt.show()

## 4. Estimation par maximum de vraisemblance

On minimise $-\log L(\theta) = -\sum_i \log f(x_i ; \theta)$ avec L-BFGS-B, chaque paramètre étant
borné dans $[10^{-6} ; 10^{3}]$ (comme dans `mle(..., lower, upper)`).

- **Diagnostic du point de départ :** si la densité y est $\le 0$ ou non définie, on s'arrête
  (le `break` du script R, qui provoquait une erreur hors boucle, devient un `return None`).
- **Ajout :** les écarts-types sont calculés à partir de l'inverse de la hessienne de $-\log L$,
  comme le fait `vcov()` sur un objet `mle` en R. La hessienne est obtenue par différences finies.

In [ ]:
def hessienne(f, x, eps=1e-4):
    # Hessienne numérique par différences finies centrées
    x = np.asarray(x, dtype=float)
    d = len(x)
    H = np.zeros((d, d))
    for i in range(d):
        for j in range(i, d):
            ei = np.zeros(d); ei[i] = eps * max(1, abs(x[i]))
            ej = np.zeros(d); ej[j] = eps * max(1, abs(x[j]))
            H[i, j] = (f(x + ei + ej) - f(x + ei - ej) - f(x - ei + ej) + f(x - ei - ej)) / (4 * ei[i] * ej[j])
            H[j, i] = H[i, j]
    return H


def estimationPTLEN(data, start_point, verbose=True):
    # start_point : dictionnaire {m, n, p, lam, beta, c, k}
    x0 = np.array(list(start_point.values()), dtype=float)

    # Diagnostic du point de départ (log-densité finie <=> densité > 0 et définie)
    logpdf_vals = burrPTLENlogpdf(data, *x0)
    if np.any(~np.isfinite(logpdf_vals)):
        print("Diagnostic : certaines valeurs du PDF sont <= 0 ou NA aux paramètres initiaux.\n"
              "Revoir votre point initial")
        return None

    # Moins la log-vraisemblance, calculée avec la log-densité stable
    def log_likelihood(theta):
        logpdf_vals = burrPTLENlogpdf(data, *theta)
        # Si la densité est nulle, NA ou infinie, la log-vraisemblance est indéfinie : forte pénalité
        if np.any(~np.isfinite(logpdf_vals)):
            return 1e9
        return -np.sum(logpdf_vals)

    try:
        res = optimize.minimize(log_likelihood, x0, method="L-BFGS-B",
                                bounds=[(1e-6, 1e3)] * 7)
    except Exception as e:
        print("Erreur lors de l'estimation MLE avec L-BFGS-B :", e)
        return None

    # Écarts-types à partir de la hessienne (si elle est inversible et définie positive)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        H = hessienne(log_likelihood, res.x)
        try:
            vcov = np.linalg.inv(H)
            se = np.sqrt(np.diag(vcov))
        except np.linalg.LinAlgError:
            se = np.full(7, np.nan)

    if verbose:
        print("Convergence :", res.success, "-", res.message)
        print(f"-log L = {res.fun:.4f}")
    return {"estimate": res.x, "se": se, "minuslogl": res.fun, "result": res}

In [ ]:
# Estimation en partant des vraies valeurs
fit = estimationPTLEN(simulation1, start_point=theta_vrai)

print(f"\n{'paramètre':>10} {'vrai':>8} {'estimé':>10} {'écart-type':>11}")
for nom, vrai, est, se in zip(NOMS, theta_vrai.values(), fit["estimate"], fit["se"]):
    print(f"{nom:>10} {vrai:>8.3f} {est:>10.4f} {se:>11.4f}")

theta_hat = fit["estimate"]

**À savoir :** avec 7 paramètres et 250 observations, plusieurs combinaisons de paramètres donnent des
courbes presque identiques (problème d'**identifiabilité**). Les estimations peuvent donc s'éloigner des
vraies valeurs, ou finir sur une borne (par exemple $\lambda \approx 10^{-6}$), tout en ajustant très bien
les données. Les écarts-types sont alors grands, voire `nan` si la hessienne n'est pas inversible.

## 5. Tests d'adéquation (ajout)

On teste $H_0$ : « les données suivent la loi Burr-PTLEN de paramètres $\hat\theta$ ».
Les trois statistiques comparent la fonction de répartition empirique $F_n$ à la loi ajustée $\hat F$,
avec $z_i = \hat F(x_{(i)})$ pour les observations triées :

| Test | Statistique | Sensibilité |
|---|---|---|
| Kolmogorov-Smirnov (KS) | $D = \sup_x \lvert F_n(x) - \hat F(x)\rvert$ | plus grand écart, surtout au centre |
| Cramér-von Mises (CvM) | $W^2 = \frac{1}{12n} + \sum_i \big(z_i - \frac{2i-1}{2n}\big)^2$ | écart quadratique moyen |
| Anderson-Darling (AD) | $A^2 = -n - \frac{1}{n}\sum_i (2i-1)\big[\ln z_i + \ln(1 - z_{n+1-i})\big]$ | donne plus de poids aux **queues** |

- KS et CvM : `scipy.stats.kstest` et `scipy.stats.cramervonmises`, qui donnent aussi la p-value.
- AD : SciPy ne le propose pas pour une loi quelconque. On calcule $A^2$ directement, avec la p-value
  asymptotique de Marsaglia & Marsaglia (2004, *Journal of Statistical Software*).

**Règle de décision :** si la p-value est supérieure à 5 %, on ne rejette pas $H_0$ : la loi ajustée est compatible avec les données.

In [ ]:
def anderson_darling(data, cdf):
    # Statistique A² d'Anderson-Darling pour une loi continue entièrement spécifiée
    z = np.clip(np.sort(cdf(data)), 1e-12, 1 - 1e-12)
    n = len(z)
    i = np.arange(1, n + 1)
    return -n - np.mean((2 * i - 1) * (np.log(z) + np.log(1 - z[::-1])))


def ad_pvalue_asymptotique(A2):
    # P(A² > a) asymptotique, approximation de Marsaglia & Marsaglia (2004)
    z = A2
    if z < 2:
        cdf = np.exp(-1.2337141 / z) / np.sqrt(z) * (2.00012 + (0.247105 - (0.0649821 - (0.0347962
              - (0.011672 - 0.00168691 * z) * z) * z) * z) * z)
    else:
        cdf = np.exp(-np.exp(1.0776 - (2.30695 - (0.43424 - (0.082433 - (0.008056
              - 0.0003146 * z) * z) * z) * z) * z))
    return 1 - cdf


def tests_adequation(data, theta):
    F_hat = lambda x: burrPTLENcdf(x, *theta)    # loi ajustée
    ks = stats.kstest(data, F_hat)
    cvm = stats.cramervonmises(data, F_hat)
    A2 = anderson_darling(data, F_hat)
    return {
        "KS":  (ks.statistic, ks.pvalue),
        "CvM": (cvm.statistic, cvm.pvalue),
        "AD":  (A2, ad_pvalue_asymptotique(A2)),
    }


resultats = tests_adequation(simulation1, theta_hat)

print(f"{'test':>5} {'statistique':>12} {'p-value':>9}   décision (5 %)")
for nom, (stat, pval) in resultats.items():
    decision = "on ne rejette pas H0" if pval > 0.05 else "on rejette H0"
    print(f"{nom:>5} {stat:>12.4f} {pval:>9.4f}   {decision}")

Comparaison visuelle (ajout) : fonction de répartition empirique et fonction de répartition ajustée.
Ce sont ces deux courbes que les trois tests comparent.

In [ ]:
x_tries = np.sort(simulation1)
F_emp = np.arange(1, len(x_tries) + 1) / len(x_tries)
grille = np.linspace(0, x_tries.max(), 400)

plt.figure(figsize=(8, 5))
plt.step(x_tries, F_emp, where="post", label="répartition empirique $F_n$")
plt.plot(grille, burrPTLENcdf(grille, *theta_hat), "r", lw=2, label=r"répartition ajustée $\hat F$")
plt.xlabel("x"); plt.ylabel("F(x)")
plt.legend()
plt.show()

## 6. Attention : paramètres estimés et p-values

Les p-values de la section 5 supposent que la loi testée est **connue à l'avance**. Ici, $\hat\theta$ a été
estimé **sur les mêmes données** : la loi ajustée « colle » aux données et les statistiques sont
plus petites qu'elles ne devraient l'être. Les p-values classiques sont donc **trop grandes**
(le test est conservateur).

La solution standard est le **bootstrap paramétrique** :

1. simuler $B$ échantillons de même taille selon la loi ajustée $\hat\theta$ ;
2. sur chacun, **ré-estimer** les paramètres puis recalculer les statistiques KS, CvM et AD ;
3. p-value = proportion des statistiques bootstrap supérieures ou égales à la statistique observée.

Chaque ré-estimation prend moins d'une seconde : avec $B = 500$, compter environ 7 minutes.
Augmenter $B$ (1000 ou plus) donne des p-values plus précises.

In [ ]:
def pvalues_bootstrap(data, theta_hat, stats_obs, B=500):
    n = len(data)
    start = dict(zip(["m", "n", "p", "lam", "beta", "c", "k"], theta_hat))
    stats_boot = {nom: [] for nom in stats_obs}

    for b in range(B):
        # 1. Échantillon simulé selon la loi ajustée (quantile exact : rapide)
        x_b = burrPTLENquantile(rng.uniform(size=n), *theta_hat)
        if not np.all(np.isfinite(x_b)):
            continue
        # 2. Ré-estimation puis statistiques sur l'échantillon simulé
        fit_b = estimationPTLEN(x_b, start_point=start, verbose=False)
        if fit_b is None:
            continue
        res_b = tests_adequation(x_b, fit_b["estimate"])
        for nom in stats_obs:
            stats_boot[nom].append(res_b[nom][0])

    # 3. p-value = proportion de statistiques bootstrap >= statistique observée
    pvals = {nom: np.mean(np.array(stats_boot[nom]) >= stats_obs[nom]) for nom in stats_obs}
    return pvals, len(stats_boot["KS"])


stats_obs = {nom: stat for nom, (stat, _) in resultats.items()}
with warnings.catch_warnings():
    warnings.simplefilter("ignore")   # avertissements numériques sans conséquence pendant les ré-estimations
    pvals_boot, B_valides = pvalues_bootstrap(simulation1, theta_hat, stats_obs, B=500)

print(f"Réplications bootstrap valides : {B_valides}\n")
print(f"{'test':>5} {'statistique':>12} {'p-value classique':>18} {'p-value bootstrap':>18}")
for nom, (stat, pval) in resultats.items():
    print(f"{nom:>5} {stat:>12.4f} {pval:>18.4f} {pvals_boot[nom]:>18.4f}")

## Conclusion

- La simulation par inversion est validée : histogramme et densité théorique se superposent, et la
  bissection coïncide avec le quantile exact.
- L'estimation par maximum de vraisemblance ajuste bien les données, mais certains des 7 paramètres
  sont mal identifiés.
- Les tests KS, CvM et AD mesurent l'écart entre la loi ajustée et les données. Une p-value supérieure à 5 %
  signifie qu'on ne rejette pas l'adéquation. Comme les paramètres sont estimés sur les mêmes données,
  ce sont les **p-values bootstrap** qu'il faut retenir.